## NB03 Analysis

In [23]:
import pandas as pd 
import os
from pathlib import Path
import sqlite3
import re

# Visualise the distribution of comments per post
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm
from datetime import datetime
from sqlalchemy import create_engine, text

# Import our custom Reddit API module

# --- Configuration for Jupyter ---
# The following magic command is for Jupyter notebooks to render plots inline.
# It should be commented out when running as a standalone script.
%config InlineBackend.figure_formats = ['svg']


In [24]:
engine = create_engine("sqlite:///../data/fantasy_data.db")

,MAX(roundId)
0,16


1. Which Players Delivered the Most Value

Focusing on Total Points compared to where they were drafted 

In [ ]:
def calculate_all_vorp(engine):
    query = """
    WITH 
    qb_replacement AS (
        SELECT AVG(avg_points) AS qb_ppg
        FROM players
        WHERE position = 'QB' AND posRank BETWEEN 15 AND 18
    ),
    rb_replacement AS (
        SELECT AVG(avg_points) AS rb_ppg
        FROM players
        WHERE position = 'RB' AND posRank BETWEEN 29 AND 32
    ),
    wr_replacement AS (
        SELECT AVG(avg_points) AS wr_ppg
        FROM players
        WHERE position = 'WR' AND posRank BETWEEN 29 AND 32
    ),
    te_replacement AS (
        SELECT AVG(avg_points) AS te_ppg
        FROM players
        WHERE position = 'TE' AND posRank BETWEEN 15 AND 16
    ),
    k_replacement AS (
        SELECT AVG(avg_points) AS k_ppg
        FROM players
        WHERE position = 'K' AND posRank BETWEEN 15 AND 18
    ),
    dst_replacement AS (
        SELECT AVG(avg_points) AS dst_ppg
        FROM players
        WHERE position = 'D/ST' AND posRank BETWEEN 15 AND 18
    )

    SELECT 
        d.player_id,
        p.player_name,
        p.position,
        p.avg_points,
        d.overallPickNumber,
        d.roundId,
        CASE 
            WHEN p.position = 'QB' THEN p.avg_points - (SELECT ppg FROM qb_replacement)
            WHEN p.position = 'RB' THEN p.avg_points - (SELECT ppg FROM rb_replacement)
            WHEN p.position = 'WR' THEN p.avg_points - (SELECT ppg FROM wr_replacement)
            WHEN p.position = 'TE' THEN p.avg_points - (SELECT ppg FROM te_replacement)
            WHEN p.position = 'K' THEN p.avg_points - (SELECT ppg FROM k_replacement)
            WHEN p.position = 'D/ST' THEN p.avg_points - (SELECT ppg FROM dst_replacement)
            ELSE NULL
        END AS vorp
    FROM draft d
    LEFT JOIN players p ON d.player_id = p.player_id
    LEFT JOIN qb_replacement qb
    LEFT JOIN rb_replacement rb 
    LEFT JOIN wr_replacement wr
    LEFT JOIN te_replacement te 
    LEFT JOIN k_replacement k 
    LEFT JOIN dst_replacement dst 
    WHERE p.position IN ('QB', 'RB', 'WR', 'TE', 'K', 'D/ST') AND p.avg_points IS NOT NULL
    ORDER BY vorp DESC
    """
    return pd.read_sql(query, engine)


In [83]:
draft_df = calculate_all_vorp(engine)
draft_df

OperationalError: (sqlite3.OperationalError) row value misused
[SQL: 
    WITH 
    qb_replacement AS (
        SELECT AVG(avg_points) AS qb_ppg
        FROM players
        WHERE position = 'QB' AND posRank BETWEEN 15 AND 18
    ),
    rb_replacement AS (
        SELECT AVG(avg_points) AS rb_ppg
        FROM players
        WHERE position = 'RB' AND posRank BETWEEN 29 AND 32
    ),
    wr_replacement AS (
        SELECT AVG(avg_points) AS wr_ppg
        FROM players
        WHERE position = 'WR' AND posRank BETWEEN 29 AND 32
    ),
    te_replacement AS (
        SELECT AVG(avg_points) AS te_ppg
        FROM players
        WHERE position = 'TE' AND posRank BETWEEN 15 AND 16
    ),
    k_replacement AS (
        SELECT AVG(avg_points) AS k_ppg
        FROM players
        WHERE position = 'K' AND posRank BETWEEN 15 AND 18
    ),
    dst_replacement AS (
        SELECT AVG(avg_points) AS dst_ppg
        FROM players
        WHERE position = 'D/ST' AND posRank BETWEEN 15 AND 18
    )

 

In [37]:
query = """
    WITH player_ppg AS (
    SELECT 
        player_id,
        position,
        points,
        games_played,
        (points * 1.0 / games_played) AS ppg
    FROM players
    WHERE games_played > 0 AND points IS NOT NULL
),

ranked_pos AS (
    SELECT *,
        RANK() OVER (PARTITION BY position ORDER BY ppg DESC) AS pos_rank
    FROM player_ppg
),

replacement_level AS (
    SELECT 
        position,
        AVG(ppg) AS replacement_ppg
    FROM ranked_pos
    WHERE pos_rank BETWEEN 15 AND 20 -- customize this range
    GROUP BY position
)

SELECT 
    r.player_id,
    r.position,
    r.ppg,
    p.player_name,
    rep.replacement_ppg,
    ROUND(r.ppg - rep.replacement_ppg, 2) AS vorp
FROM ranked_pos r
JOIN replacement_level rep ON r.position = rep.position
WHERE r.ppg IS NOT NULL
ORDER BY vorp DESC
    """
df = pd.read_sql(query, engine)
df


OperationalError: (sqlite3.OperationalError) no such column: p.player_name
[SQL: 
    WITH player_ppg AS (
    SELECT 
        player_id,
        position,
        points,
        games_played,
        (points * 1.0 / games_played) AS ppg
    FROM players
    WHERE games_played > 0 AND points IS NOT NULL
),

ranked_pos AS (
    SELECT *,
        RANK() OVER (PARTITION BY position ORDER BY ppg DESC) AS pos_rank
    FROM player_ppg
),

replacement_level AS (
    SELECT 
        position,
        AVG(ppg) AS replacement_ppg
    FROM ranked_pos
    WHERE pos_rank BETWEEN 15 AND 20 -- customize this range
    GROUP BY position
)

SELECT 
    r.player_id,
    r.position,
    r.ppg,
    p.player_name,
    rep.replacement_ppg,
    ROUND(r.ppg - rep.replacement_ppg, 2) AS vorp
FROM ranked_pos r
JOIN replacement_level rep ON r.position = rep.position
WHERE r.ppg IS NOT NULL
ORDER BY vorp DESC
    ]
(Background on this error at: https://sqlalche.me/e/20/e3q8)

Focus on who was the best value in the fantasy league

In [34]:
query = """
    WITH ranked_draft AS (
        SELECT
            player_id,
            RANK() OVER (ORDER BY avg ASC) AS overallPickNumber
        FROM draft
    ),
    ranked_points AS (
        SELECT
            player_id,
            points,
            RANK() OVER (ORDER BY points DESC) AS performance_rank
        FROM players
        WHERE points IS NOT NULL
    )

    SELECT
        p.player_name,
        p.position,
        d.overallPickNumber,
        pr.performance_rank,
        pr.points,
        p.current_team_name,
        (d.overallPickNumber - pr.performance_rank) AS value_score
    FROM draft d
    JOIN ranked_points pr ON d.player_id = pr.player_id
    JOIN players p ON p.player_id = d.player_id
    ORDER BY value_score DESC
    """
df = pd.read_sql(query, engine)
df



,player_name,position,overallPickNumber,performance_rank,points,current_team_name,value_score
0,Bo Nix,BE,214,15,300,Mendes Army,199
1,Geno Smith,QB,207,32,250,Jean Machine,175
2,Baker Mayfield,QB,169,5,352,South Bay Starr Power,164
3,Chuba Hubbard,RB,202,46,232,Ambler Thighs,156
4,Brian Thomas Jr.,RB/WR/TE,154,20,275,The Ralph Dudes,134
...,...,...,...,...,...,...,...
219,Isiah Pacheco,RB,8,314,52,Deb’s Debsters,-306
220,MarShawn Lloyd,None,167,481,2,FA,-314
221,Zamir White,BE,62,391,26,Ambler Thighs,-329
222,Christian McCaffrey,None,1,337,43,FA,-336


Who had the most value per ppg

In [19]:
pd.read_sql("SELECT * FROM players LIMIT 5", engine)
pd.read_sql("SELECT * FROM average_draft_position LIMIT 5", engine)
pd.read_sql("SELECT * FROM draft LIMIT 5", engine)


,player_id,overallPickNumber,team_id,roundPickNumber,id,roundId,autoDraftTypeId,lineupSlotId
0,3117251,1,3,1,1,1,0,2
1,4241389,2,14,2,2,1,0,4
2,3929630,3,1,3,3,1,0,2
3,3918298,4,4,4,4,1,0,0
4,4427366,5,8,5,5,1,3,2
